# RAG(Retrival Augmented Generation)
## has four major components
## 1. Document Loader
## 2. Text Spliter
## 3. Vector Database
## 4. Retrivers


# RAG
## RAG is a technique that combine retrived information with Language Generation, where model retrive information from database and generate response.

# In langchain There are Hundreds of Document loaders Few of them are as
## TextLoaders
## PyPDFLoader
## WebBaseLoader
## CSVLoader
### Document loaders are compoenent in langchain used to load data from various sources into a standardize object (Document Object) which can then be used for chunking, embedding, retrival and generation.

### Every Document Object has two part Actual content (Page Data), and Meta data

# Text Loader
### Reads plain text from (.txt files) and convert them into Langchain Document object best for Chatlog, code snipets, Scrapped Text

In [ ]:
from gitdb.fun import chunk_size
from langchain_community.document_loaders import TextLoader

from chapters.Runnable_clonning import gemini_powered_llm


In [ ]:
Text_loader_Object = TextLoader(
    file_path= "/Users/abhisheksingh/Lang_Chain/chat_history.txt",

)

In [ ]:
doc = Text_loader_Object.load()

In [ ]:
print(doc)

In [ ]:
print(type(doc))

In [ ]:
print(doc[0])

In [ ]:
print(doc[0].page_content)

In [ ]:
print(type(doc[0]))

In [ ]:
print(doc[0].metadata)

In [ ]:
print(type(doc[0].metadata))

In [ ]:
# lets pass this content to llm
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import TextLoader
import os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=gemini_key
)
Text_loader_Object = TextLoader(
    file_path= "/Users/abhisheksingh/Lang_Chain/chat_history.txt",

)
cotent = Text_loader_Object.load()
content = cotent[0].page_content
prompt = PromptTemplate(
    template="Summarize the given text : {text}",
    input_variables=["text"]
)
chain = prompt | gemini_powered_llm | StrOutputParser()
result =chain.invoke(cotent)
print(result)

In [ ]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableSequence,RunnableParallel
from langchain_community.document_loaders import TextLoader
import  os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=gemini_key

)
prompt_1 = PromptTemplate(
    template = "Generate an Eassy on given topic :{topic} ",
    input_variables = ["topic"]
)
prompt_2 = PromptTemplate(
    template = "Summarize the given text, and prepare the post for instagram with given text : {text}",
    input_variables = ["text"]
)
prompt_3 = PromptTemplate(
    template = "Summarize the given text, and prepare the post for Linkedin with given text : {text}",
    input_variables = ["text"]
)
parser = StrOutputParser()
chain_1 = RunnableSequence(prompt_1 |gemini_powered_llm | parser)
chain_2 = RunnableParallel({
    "insta": (prompt_2 | gemini_powered_llm | parser),
    "linkedin": (prompt_3 | gemini_powered_llm | parser)
})
final_chain = RunnableSequence(chain_1,chain_2)
result = final_chain.invoke({"topic": "AI"})
print(result)

In [ ]:
print(result["insta"])

In [ ]:
print(result["linkedin"])

# PyPDFLoader
### Loads PDF files into a Document Object and convert each page into a document object
### every Document object look like a list of dictionary i.e [{"Page_content": "....", "Meta_data":".."},{"Page_content": "....", "Meta_data":".."}]

In [ ]:
document_object ="/Users/abhisheksingh/Desktop/Abhishek_Singh_Data_Engineer.pdf"

In [ ]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFLoader
from pydantic import BaseModel
from typing import List
import  os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=gemini_key
)

pdf_loader = PyPDFLoader(
    file_path=document_object,
)
loaded_pdf = pdf_loader.load()

# preparing prompts
prompt = PromptTemplate(
    template = "I am attaching my resume, resume :{resume}, Keep in mind that I am preparing for data engineering then analyse the resume and findout the tech where I am lacking and what are my strong areas also provide me summary of my resume",
    input_variables = ["resume"]
)
prompt_to_grow_weak_areas = PromptTemplate(
    template = " I want to grow my areas of expertise in the following areas : {areas}. PLease suggest me project that I can work on to build my skills in these areas",
    input_variables = ["areas"]
)

class response_of_llm(BaseModel):
    tech_areas_to_improve: list[str]
    tech_areas_already_good: list[str]
    project_to_work_on: list[str]
    summary_of_resume: str

parser_1 = PydanticOutputParser(pydantic_object=response_of_llm)
parser_2 = StrOutputParser()

chain_1 = prompt | gemini_powered_llm | parser_1

#area_to_grow = chain_1["tech_areas_to_improve"]
chain_2 = RunnableParallel({
    "project_to_work_on": (prompt_to_grow_weak_areas | gemini_powered_llm | parser_2),
    "summary_of_resume": (RunnablePassthrough() | gemini_powered_llm | parser_2)
})
final_chain = RunnableSequence(chain_1,chain_2)

result =final_chain.invoke({"resume": loaded_pdf})

In [ ]:
# To load multiple file(pdf from direcoty)
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import DirectoryLoader # use this to load multiple files
from pydantic import BaseModel
from typing import List
import  os
from dotenv import load_dotenv

docs= DirectoryLoader(
    "/Users/abhisheksingh/Desktop/",
    glob="Abhishek*.pdf",
    loader_cls=PyPDFLoader
)
loaded_pdf = docs.load()
print(loaded_pdf)

In [ ]:
print(len(loaded_pdf))

In [ ]:
loaded_pdf[0].metadata



# Lazy_Loader
### Loads files from a directory and convert them into Document Object, one by one very usefull incase of large number of documents and for streaming purpose.

# WebBaseLoader
### Loads webpages into Document Object

In [ ]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFLoader,WebBaseLoader
from langchain_community.document_loaders import DirectoryLoader # use this to load multiple files
from pydantic import BaseModel
from typing import List
import  os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=gemini_key
)
url = "https://www.flipkart.com/apple-macbook-air-m5-2026-m5-16-gb-512-gb-ssd-tahoe-mdhe4hn-a/p/itm8505e2f874525?pid=COMHH78YEUAMB68W&lid=LSTCOMHH78YEUAMB68WGNHGES&marketplace=FLIPKART&q=macbook&store=6bo%2Fb5g&srno=s_1_1&otracker=AS_QueryStore_OrganicAutoSuggest_2_4_na_na_na&otracker1=AS_QueryStore_OrganicAutoSuggest_2_4_na_na_na&fm=organic&iid=50fddef3-5b8f-41c3-ab39-2742014eae25.COMHH78YEUAMB68W.SEARCH&ppt=None&ppn=None&ssid=6d5oz8r05e9r8lxc1777847757253&qH=864faee128623e2f&ov_redirect=true"
webloader = WebBaseLoader(
    url,

)
loaded_web = webloader.load()
print(loaded_web)


In [ ]:
print(loaded_web[0].metadata)

In [ ]:
print(loaded_web[0].page_content)

# Text Spliter
## Text spliting is a process of breaking Large text object into smaller chunk(Like on basis of Paragraph, Pages) that a LLm can handle eaisly.
### Why text spliting was required, Because we have limitation over the Context length of LLMs.
### Help to improve downstreaming task (Embedding, Scymentic search, Summarization).
### We gat better embedded vectors if we do embedding after splittings. Similarly with Summarization also works better with small text and same with symentic search.


# Text Splitter on the basis of
## Length Based
## Text Structured Based
## Document Structured Based
## Semantic Meaning Based


# Length Based Text Spliting

## based on length, based on tokens


In [75]:
from langchain_text_splitters import  CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=False,
)

In [76]:
text = "Yes, your AI agent can absolutely access, read, and respond to your emails. This technology is typically referred to as an AI Email Agent, which differs from a standard writing assistant because it can autonomously monitor an inbox and take actions within defined boundaries.Setting Up an AI Email AgentSeveral approaches can enable these capabilities, depending on technical comfort and specific needs:No-Code Automation Platforms: Tools like Zapier and n8n allow for creating agents that monitor inboxes, read content, and draft or send replies based on defined rules without writing code.Built-in Enterprise Agents:Microsoft Copilot Studio: A custom agent can connect directly to Office 365 Outlook to read threads and use tools to send formatted emails.Gemini for Google Workspace: Offers native integration for Gmail users to summarize long threads and draft context-aware responses.Developer-Focused APIs: Platforms like AgentMail provide an Email Inbox API specifically for AI agents, allowing them to have their own inboxes to receive verification codes, parse attachments, and send programmatic emails.Key CapabilitiesA configured email agent can perform the following tasks:Triage and Classification: Automatically label and route incoming emails to specific folders or team members based on intent.Autonomous Action: Execute tasks like creating a calendar event, checking shipment status in a CRM, or triggering a password reset.Human-in-the-Loop: Set permissions so the agent only creates drafts for review before sending."

In [77]:
result =text_splitter.split_text(text)

In [78]:
print(result)

['Yes, your AI agent can absolutely access, read, and respond to your emails. This technology is typically referred to as an AI Email Agent, which differs from a standard writing assistant because it can autonomously monitor an inbox and take actions within defined boundaries.Setting Up an AI Email AgentSeveral approaches can enable these capabilities, depending on technical comfort and specific needs:No-Code Automation Platforms: Tools like Zapier and n8n allow for creating agents that monitor inboxes, read content, and draft or send replies based on defined rules without writing code.Built-in Enterprise Agents:Microsoft Copilot Studio: A custom agent can connect directly to Office 365 Outlook to read threads and use tools to send formatted emails.Gemini for Google Workspace: Offers native integration for Gmail users to summarize long threads and draft context-aware responses.Developer-Focused APIs: Platforms like AgentMail provide an Email Inbox API specifically for AI agents, allowi

In [98]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import os
from dotenv import load_dotenv
load_dotenv()

gemini_powered_llm = GoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=os.getenv("GEMINI_API_KEY")
)
docs= DirectoryLoader(
    "/Users/abhisheksingh/Desktop/",
    glob="Abhishek*.pdf",
    loader_cls=PyPDFLoader
)
loaded_doc = docs.load()
text = loaded_doc[0].page_content
text_splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=" "

)
splited_text = text_splitter.split_text(text
                                        )
print(splited_text)
print(len(splited_text))

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)


['Abhishek Singh Data Engineer Mob-7903242876,  | email- 20mecp01@iiitdmj.ac.in| github- h@ps://github.com/CodeStarkz Tableau-h@ps://public.tableau.com/app/proﬁle/abhishek.singh8569/vizzes \nPROFILE Technical Support Engineer with 2.5+ years of experience in tech support, system monitoring, and incident resoluOon. Proﬁcient in providing technical and operaOonal support for B2B customers, with hands-on experience troubleshooOng distributed network issues, analyzing system logs, and opOmizing plaUorm performance. Skilled in Salesforce/ServiceNow GckeGng, SQL (CRUD operaGons, CTEs, window funcGons), and Linux/Unix environments. And well sound knowledge and experience of about API tesGng in Postman, and containerisaGon with Docker. Recently obtained the cerGﬁcaGon in Advanced Excel. \nStrong programming knowledge in Python, leveraging object-oriented, Data Structure & Algorithms principles to automate repeOOve support tasks and analyze operaOonal data. Experienced with Python libraries suc

In [100]:
print(loaded_doc[1].metadata)

{'producer': 'macOS Version 12.7.6 (Build 21H1320) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260329194648Z00'00'", 'title': 'Abhishek_Singh_Data_Engineer', 'moddate': "D:20260329194648Z00'00'", 'source': '/Users/abhisheksingh/Desktop/Abhishek_Singh_Data_Engineer.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}


In [101]:
print(len(loaded_doc))

4


# Text Structure Based TextSpliting
